In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch
import numpy as np

import gdown
import zipfile


2025-04-10 16:50:22.910294: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744303823.153099      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744303823.221012      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
output_zip = "nli_roberta_model.zip"
extract_to = "nli_roberta_model"

url = f"https://drive.google.com/uc?id=1F-QLA-fLmYIzxXsOlh5RnlNx3A6WydL1"

# Download the zip file
gdown.download(url, output=output_zip, quiet=False)

# Unzip the contents
with zipfile.ZipFile(output_zip, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print(f"Model extracted to: {extract_to}")


Downloading...
From (original): https://drive.google.com/uc?id=1F-QLA-fLmYIzxXsOlh5RnlNx3A6WydL1
From (redirected): https://drive.google.com/uc?id=1F-QLA-fLmYIzxXsOlh5RnlNx3A6WydL1&confirm=t&uuid=35a99f65-da6e-45ad-b452-ec3dd5a18354
To: /kaggle/working/nli_roberta_model.zip
100%|██████████| 1.30G/1.30G [00:07<00:00, 175MB/s]


Model extracted to: nli_roberta_model


In [3]:
model = AutoModelForSequenceClassification.from_pretrained("nli_roberta_model")
tokenizer = AutoTokenizer.from_pretrained("nli_roberta_model")
model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=Tru

In [4]:
df_test = pd.read_csv("/kaggle/input/nli-test-dataset/test.csv")
ds_test = Dataset.from_pandas(df_test)

def tokenize(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True, padding="max_length", max_length=256)




In [5]:
ds_test = ds_test.map(tokenize, batched=True)

training_args = TrainingArguments(
    output_dir="./tmp",
    per_device_eval_batch_size=8,
    report_to="none"
)
trainer = Trainer(model=model,
                 args=training_args)


Map:   0%|          | 0/3302 [00:00<?, ? examples/s]

In [6]:
preds = trainer.predict(ds_test)
labels = np.argmax(preds.predictions, axis=1)

In [7]:
output = pd.DataFrame({"prediction": labels})
output.to_csv("Group_52_C.csv", index=False)